# Weather Prediction Pipeline — Data Ingestion (Databricks)

Loads the two data sources from the project proposal:

1. **NOAA CDO (Climate Data Online)** — historical daily station data, used for training/test data. Requires a free API token.
2. **api.weather.gov (NWS)** — live/current observations and forecasts, no key required. Used later for the production/monitoring and drift-simulation stages.

> This notebook only covers **Stage 1: Data Ingestion & Baselines**. Preprocessing, MLflow tracking, etc. live in separate notebooks per the project outline.

## Prerequisites (one-time, before running this notebook)

1. Get a free NOAA CDO token: https://www.ncdc.noaa.gov/cdo-web/token
2. Store it in a Databricks secret scope (never hardcode it in the notebook):
   ```bash
   databricks secrets create-scope weather-mlops
   databricks secrets put-secret weather-mlops noaa_cdo_token
   ```
3. Attach this notebook to a cluster/SQL warehouse with Unity Catalog access if you want the Delta writes in section 4 to succeed, and update the `catalog` widget if you're not using `main`.

All other run-time parameters (location, date range, coordinates, catalog/schema) are exposed as notebook widgets in section 0 so this can be scheduled as a parameterized Databricks Job/Workflow task.</cell_id>
</invoke>


## 0. Setup

In [0]:
import time
import requests
import pandas as pd
from datetime import datetime, timedelta

pd.set_option('display.max_columns', None)

# Notebook widgets: configurable per-run (or via a Databricks Job/Workflow task's parameters)
dbutils.widgets.text("location_id", "FIPS:17031", "NOAA locationid")
dbutils.widgets.text("start_date", "2020-01-01", "Start date")
dbutils.widgets.text("end_date", "2024-12-31", "End date")
dbutils.widgets.text("lat", "41.85", "NWS latitude")
dbutils.widgets.text("lon", "-87.65", "NWS longitude")
dbutils.widgets.text("nws_contact_email", "klkendall@uchicago.edu", "Contact email for NWS User-Agent")
dbutils.widgets.text("catalog", "mlo", "Unity Catalog catalog")
dbutils.widgets.text("schema", "weather_mlops", "Schema for bronze tables")


## 1. NOAA CDO — Historical Data

Get a free token here: https://www.ncdc.noaa.gov/cdo-web/token

**Do not hardcode your token in the notebook.** It's read from the Databricks secret scope configured via the `secret_scope` / `secret_key` widgets above (see the Prerequisites section for how to create it).

In [0]:
NOAA_TOKEN = api_key = dbutils.secrets.get(scope="mlo", key="WEATHER_API_KEY")

NOAA_BASE_URL = "https://www.ncei.noaa.gov/cdo-web/api/v2"
HEADERS = {"token": NOAA_TOKEN}


### 1.1 Find a station


In [0]:
# Chicago, IL -> FIPS location id 'FIPS:17031' (Cook County) by default. Override via the location_id widget.
LOCATION_ID = dbutils.widgets.get("location_id")

print(LOCATION_ID)

def get_stations(location_id, datasetid="GHCND", limit=1000):
    resp = requests.get(
        f"{NOAA_BASE_URL}/stations",
        headers=HEADERS,
        params={"locationid": location_id, "datasetid": datasetid, "limit": limit},
    )
    resp.raise_for_status()
    return pd.DataFrame(resp.json().get("results", []))

stations_df = get_stations(LOCATION_ID)
stations_df[["id", "name", "mindate", "maxdate", "datacoverage"]].head(10) if not stations_df.empty else stations_df


### 1.2 Pull historical daily data (GHCND)

`datasetid=GHCND` (Global Historical Climatology Network - Daily). Common `datatypeid` values: `TMAX`, `TMIN`, `PRCP`, `AWND`, `SNOW`.

The API caps each request to a **1-year date range and 1000 records**, so historical pulls are paginated by year below.

In [0]:
NOAA_MIN_REQUEST_INTERVAL = 0.25  # NOAA CDO caps requests at ~5/sec; stay comfortably under that
NOAA_MAX_RETRIES = 5

def _noaa_get(url, params):
    """GET with NOAA-friendly pacing and retry/backoff on 429s."""
    for attempt in range(NOAA_MAX_RETRIES):
        resp = requests.get(url, headers=HEADERS, params=params)
        time.sleep(NOAA_MIN_REQUEST_INTERVAL)
        if resp.status_code == 429:
            retry_after = resp.headers.get("Retry-After")
            wait = float(retry_after) if retry_after else 2 ** attempt
            time.sleep(wait)
            continue
        resp.raise_for_status()
        return resp
    resp.raise_for_status()
    return resp


def get_ghcnd_daily(station_id, start_date, end_date, datatypes=("TMAX", "TMIN", "PRCP", "AWND")):
    """Fetch daily GHCND records for one station between start_date and end_date (YYYY-MM-DD), 
    paginated in <=1-year chunks per NOAA CDO API limits."""
    all_results = []
    chunk_start = datetime.fromisoformat(start_date)
    final_end = datetime.fromisoformat(end_date)

    while chunk_start <= final_end:
        chunk_end = min(chunk_start + timedelta(days=364), final_end)
        offset = 1
        while True:
            resp = _noaa_get(
                f"{NOAA_BASE_URL}/data",
                params={
                    "datasetid": "GHCND",
                    "stationid": station_id,
                    "datatypeid": ",".join(datatypes),
                    "startdate": chunk_start.strftime("%Y-%m-%d"),
                    "enddate": chunk_end.strftime("%Y-%m-%d"),
                    "units": "metric",
                    "limit": 1000,
                    "offset": offset,
                },
            )
            payload = resp.json()
            results = payload.get("results", [])
            all_results.extend(results)
            if len(results) < 1000:
                break
            offset += 1000
        chunk_start = chunk_end + timedelta(days=1)

    return pd.DataFrame(all_results)


In [0]:
# Pick a station whose coverage actually spans the date range we want to pull,
# instead of blindly taking the first result (which may be a long-closed station).
START_DATE, END_DATE = dbutils.widgets.get("start_date"), dbutils.widgets.get("end_date")

candidates = stations_df[
    (stations_df["mindate"] <= START_DATE) & (stations_df["maxdate"] >= END_DATE)
].sort_values("datacoverage", ascending=False)

if candidates.empty:
    raise RuntimeError(
        f"No station in stations_df covers {START_DATE}..{END_DATE}. "
        "Inspect stations_df[['id','name','mindate','maxdate']] and either "
        "pick a different location_id widget value or narrow start_date/end_date to a range "
        "a station actually covers."
    )

STATION_ID = candidates.iloc[0]["id"]
print("Using station:", STATION_ID, candidates.iloc[0]["name"])

noaa_raw = get_ghcnd_daily(
    station_id=STATION_ID,
    start_date=START_DATE,
    end_date=END_DATE,
)

noaa_raw.head()


In [0]:
# Reshape from long (one row per date/datatype) to wide (one row per date, one column per datatype)
noaa_daily = (
    noaa_raw
    .assign(date=lambda d: pd.to_datetime(d["date"]))
    .pivot_table(index=["station", "date"], columns="datatype", values="value", aggfunc="first")
    .reset_index()
    .sort_values("date")
)

noaa_daily.head()


## 2. api.weather.gov — Live Data

No API key required — only a descriptive `User-Agent` header (NWS policy). This will later feed the production inference + drift-simulation stages, but we pull a sample here to confirm access and shape.

In [0]:
NWS_HEADERS = {
    "User-Agent": f"(mlops-weather-project, contact: {dbutils.widgets.get('nws_contact_email')})",
    "Accept": "application/geo+json",
}

# Same Chicago-area point used above, as lat/lon. Override via the lat / lon widgets.
LAT, LON = float(dbutils.widgets.get("lat")), float(dbutils.widgets.get("lon"))

def get_nws_point_metadata(lat, lon):
    resp = requests.get(f"https://api.weather.gov/points/{lat},{lon}", headers=NWS_HEADERS)
    resp.raise_for_status()
    return resp.json()

point_meta = get_nws_point_metadata(LAT, LON)
forecast_url = point_meta["properties"]["forecast"]
forecast_hourly_url = point_meta["properties"]["forecastHourly"]
stations_url = point_meta["properties"]["observationStations"]
forecast_url, forecast_hourly_url, stations_url


In [0]:
def get_nws_forecast(forecast_url):
    resp = requests.get(forecast_url, headers=NWS_HEADERS)
    resp.raise_for_status()
    periods = resp.json()["properties"]["periods"]
    # json_normalize flattens nested fields (dewpoint, relativeHumidity, probabilityOfPrecipitation
    # come back as {"unitCode": ..., "value": ...} objects) into dot-separated columns so the
    # result is Spark/Delta-friendly downstream.
    return pd.json_normalize(periods)

nws_forecast = get_nws_forecast(forecast_url)
nws_forecast[["name", "startTime", "temperature", "temperatureUnit", "windSpeed", "shortForecast"]].head(10)


In [0]:
def get_nws_latest_observation(stations_url):
    stations_resp = requests.get(stations_url, headers=NWS_HEADERS)
    stations_resp.raise_for_status()
    nearest_station_id = stations_resp.json()["features"][0]["properties"]["stationIdentifier"]

    obs_resp = requests.get(
        f"https://api.weather.gov/stations/{nearest_station_id}/observations/latest",
        headers=NWS_HEADERS,
    )
    obs_resp.raise_for_status()
    return nearest_station_id, obs_resp.json()["properties"]

nearest_station_id, latest_obs = get_nws_latest_observation(stations_url)
print("Nearest station:", nearest_station_id)
pd.json_normalize(latest_obs)[["timestamp", "temperature.value", "windSpeed.value", "relativeHumidity.value", "textDescription"]]


## 3. Sanity checks

Quick shape/null checks before moving to preprocessing.

In [0]:
print("NOAA historical daily shape:", noaa_daily.shape)
print(noaa_daily.isna().mean().round(3))
print()
print("NWS forecast periods shape:", nws_forecast.shape)


In [0]:
%sql
CREATE CATALOG IF NOT EXISTS mlo;

## 4. Save raw pulls as Delta tables

Persist raw pulls as managed Delta tables (bronze layer) before any cleaning/versioning happens downstream (e.g. in the feature-store step). Target catalog/schema are controlled by the `catalog` / `schema` widgets.

In [0]:
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")

print(CATALOG)

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

noaa_table = f"{CATALOG}.{SCHEMA}.noaa_historical_daily"
nws_table = f"{CATALOG}.{SCHEMA}.nws_forecast_snapshot"


spark.sql(f"DROP TABLE IF EXISTS {noaa_table}")

noaa_spark_df = spark.createDataFrame(noaa_daily)
noaa_spark_df.write.mode("overwrite").saveAsTable(noaa_table)

nws_spark_df = spark.createDataFrame(nws_forecast)
nws_spark_df.write.mode("overwrite").saveAsTable(nws_table)

print(f"Saved NOAA historical daily -> {noaa_table} ({noaa_spark_df.count()} rows)")
print(f"Saved NWS forecast snapshot -> {nws_table} ({nws_spark_df.count()} rows)")


---
**Next steps** (per the project outline):
- Schedule this notebook as a Databricks Job/Workflow task (parameterized via the widgets defined in section 0) for recurring ingestion
- Define target variable + train/test split (Stage 1 continued), reading from the `noaa_historical_daily` bronze table
- Track dataset version alongside MLflow experiment runs

In [0]:
spark.table(noaa_table).printSchema()

In [0]:
%sql
SELECT * FROM mlo.weather_mlops.noaa_historical_daily
LIMIT 5;